# Class 4. Classical ML Refresher Through a Quantum Lens (Exercises)

EVA: Quantum Machine Learning | ZHAW | Pavel Sulimov

---

Goals of this practice session:

1. Use matrix-form notation for linear and logistic models.
2. Understand why XOR needs a feature map.
3. Distinguish an explicit XOR lift from a degree-2 polynomial kernel feature map.
4. Compute and interpret Gram matrices and kernel-derived distances.
5. Compare linear and nonlinear kernel baselines before moving to quantum feature maps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

---
## Part 1: Math tasks

### M4.1. Matrix-form prediction and sigmoid

Given
\[
X=\begin{bmatrix}
1 & 0.5\\
0.2 & 1.2\\
1.5 & 1.0
\end{bmatrix},
\quad
w=\begin{bmatrix}1.1\\-0.8\end{bmatrix},
\quad b=-0.1,
\]
compute $z=Xw+b\mathbf{1}$ and $\sigma(z)$.

In [ ]:
X = np.array([
    [1.0, 0.5],
    [0.2, 1.2],
    [1.5, 1.0],
])
w = np.array([1.1, -0.8])
b = -0.1

z = ...  # YOUR CODE HERE
sigma = ...  # YOUR CODE HERE

print("z:", np.round(z, 6))
print("sigma(z):", np.round(sigma, 6))

### M4.2. XOR lifting and separability check

Use
\(
\phi(x_1,x_2)=(x_1,x_2,x_1x_2)
\)
and verify that the hyperplane
\(
s=x_1+x_2-2x_3-0.5
\)
separates XOR classes.

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

X_lift = ...  # YOUR CODE HERE
s = ...  # YOUR CODE HERE

print("Lifted points:\n", X_lift)
print("Signed values s:", s)
print("Class 0 signs:", s[y_xor == 0])
print("Class 1 signs:", s[y_xor == 1])

### M4.3. Explicit Gram matrix for the XOR lift

Use the manual feature map
\[
\psi(x_1,x_2)=(x_1,x_2,x_1x_2)
\]
and compute the Gram matrix
\[
G_{ij}=\langle \psi(x^{(i)}),\psi(x^{(j)})\rangle
\]
for all XOR pairs.

In [ ]:
Psi = ...  # YOUR CODE HERE
G_explicit = ...  # YOUR CODE HERE

print("Lifted feature matrix Psi:\n", Psi)
print("Explicit Gram matrix G:\n", G_explicit)

### M4.4. Polynomial kernel matrix and kernel-derived distance

Now switch to the degree-2 polynomial kernel
\[
K(x,y)=(x^\top y + 1)^2.
\]

1. Compute the full kernel matrix for the XOR points.
2. Use
\[
\|\phi(x)-\phi(y)\|^2 = K(x,x)+K(y,y)-2K(x,y)
\]
to recover the squared distance between $x=(0,1)$ and $y=(1,0)$ in the implicit feature space.

In [ ]:
K_poly = ...  # YOUR CODE HERE

d2_poly = ...  # YOUR CODE HERE

print("Polynomial kernel matrix:\n", K_poly)
print(f"Squared distance between (0,1) and (1,0): {d2_poly:.1f}")

---
## Part 2: Programming tasks

### P4.1. Linear, polynomial, and RBF SVM on two-moons

In [ ]:
X, y = make_moons(n_samples=260, noise=0.22, random_state=7)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=7, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

svc_lin = SVC(kernel="linear", C=1.0)
svc_poly = SVC(kernel="poly", C=1.0, degree=3, gamma=1.0, coef0=1.0)
svc_rbf = SVC(kernel="rbf", C=1.0, gamma=1.0)

# YOUR CODE HERE: fit all three models and compute accuracies
...

print(f"Linear SVM accuracy:      {acc_lin:.3f}")
print(f"Polynomial SVM accuracy:  {acc_poly:.3f}")
print(f"RBF SVM accuracy:         {acc_rbf:.3f}")

### P4.2. Visualize three decision boundaries

In [ ]:
def plot_boundary(ax, model, X_data, y_data, title):
    x_min, x_max = X_data[:, 0].min() - 0.5, X_data[:, 0].max() + 0.5
    y_min, y_max = X_data[:, 1].min() - 0.5, X_data[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 220),
        np.linspace(y_min, y_max, 220),
    )
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    zz = ...  # YOUR CODE HERE
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")
    ax.scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap="coolwarm", s=20)
    ax.set_title(title)
    ax.grid(True, alpha=0.2)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
# YOUR CODE HERE: call plot_boundary for linear, polynomial, and RBF models
plt.tight_layout()
plt.show()

### P4.3. Recover a feature-space distance from kernel values

Use the degree-2 polynomial kernel
\[
K(x,y)=(x^\top y + 1)^2
\]
and verify numerically that
\[
\|\phi(x)-\phi(y)\|^2 = K(x,x)+K(y,y)-2K(x,y)
\]
for two chosen points from the XOR dataset.

In [ ]:
def phi_poly(X_in):
    x1 = X_in[:, 0]
    x2 = X_in[:, 1]
    return np.column_stack([
        np.ones(len(X_in)),
        np.sqrt(2.0) * x1,
        np.sqrt(2.0) * x2,
        x1 ** 2,
        np.sqrt(2.0) * x1 * x2,
        x2 ** 2,
    ])

x_i = X_xor[[1]]
x_j = X_xor[[2]]

k_ii = ...  # YOUR CODE HERE
k_jj = ...  # YOUR CODE HERE
k_ij = ...  # YOUR CODE HERE

d2_from_kernel = ...  # YOUR CODE HERE

d2_from_features = ...  # YOUR CODE HERE

print(f"K(x_i, x_i) = {k_ii:.1f}")
print(f"K(x_j, x_j) = {k_jj:.1f}")
print(f"K(x_i, x_j) = {k_ij:.1f}")
print(f"Distance^2 from kernel values:   {d2_from_kernel:.1f}")
print(f"Distance^2 from explicit phi(x): {d2_from_features:.1f}")

---
## Summary

A Gram matrix is a table of pairwise inner products in feature space. Kernels let us build that geometry without listing all coordinates explicitly, and the same kernel values determine distances in the implicit feature space.

Before moving to quantum feature maps, we should understand clearly what the classical feature map is, what similarity it induces, and how strong the classical baselines already are.